In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms, models
from PIL import Image
from tqdm.auto import tqdm

# --- 1. FILTER DATA FROM CSV ---
# UPDATE THESE PATHS TO YOUR GOOGLE DRIVE LOCATIONS
CSV_PATH = r"/content/drive/MyDrive/Major_Project/linked_egg_metadata_qa_checked.csv"
IMAGE_DIR = r"/content/drive/MyDrive/Major_Project/merged_dataset_rgb"

# Load the metadata
df = pd.read_csv(CSV_PATH)

# Strictly filter out images that failed the QA check (if the column exists)
if 'QA_Failed' in df.columns:
    clean_df = df[df['QA_Failed'] == False].copy()
else:
    clean_df = df.copy()

# Define a dictionary to map string labels to integer classes for PyTorch
# 'T' (Trống / Infertile) -> 0
# 'M' (Mái / Fertile) -> 1
LABEL_MAP = {'T': 0, 'M': 1}

# Create the data list format: [(img_path, label), ...]
data_list = []

# Create a separate list just for the labels (needed for Stratified K-Fold splitting)
all_labels = []

missing_files = 0

for _, row in clean_df.iterrows():
    # Use os.path.basename to extract just the filename (e.g., "egg_1.jpg")
    # in case the 'image_path' column contains full outdated paths.
    filename = os.path.basename(str(row['image_path']))
    img_path = os.path.join(IMAGE_DIR, filename)

    # Extract the label, strip any accidental whitespace, and convert to uppercase
    str_label = str(row['label']).strip().upper()

    # Check if the label is valid ('T' or 'M') and if the image file actually exists
    if str_label in LABEL_MAP:
        if os.path.exists(img_path):
            int_label = LABEL_MAP[str_label]

            # Append to both lists
            data_list.append((img_path, int_label))
            all_labels.append(int_label)
        else:
            missing_files += 1

print(f"[*] Successfully loaded {len(data_list)} clean images for training.")
print(f"[*] Extracted {len(all_labels)} labels for K-Fold stratification.")

if missing_files > 0:
    print(f"⚠️ Warning: {missing_files} images were listed in the CSV but not found in the directory.")

[*] Successfully loaded 4616 clean images for training.
[*] Extracted 4616 labels for K-Fold stratification.
⚠️ Warning: 2 images were listed in the CSV but not found in the directory.


In [ ]:
class SegmentedEggDataset(Dataset):
    def __init__(self, data_list, transform=None):
        self.data_list = data_list
        self.transform = transform

    def __len__(self):
        return len(self.data_list)

    def __getitem__(self, idx):
        img_path, label = self.data_list[idx]
        with Image.open(img_path) as img:
            image = img.convert('RGB')
            if self.transform:
                image = self.transform(image)
        label_tensor = torch.tensor(label, dtype=torch.long)
        return image, label_tensor

# ==========================================
# 3. TRANSFORMS, SPLITTING, AND DATALOADERS
# ==========================================
train_transforms = transforms.Compose([
    # 1. Base Resize for CNN input
    transforms.Resize((256, 224)),

    # 2. Geometric Augmentations (Simulating physical variations in candling)
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=45), # Eggs can tilt in the machine

    # Slight shift and zoom to simulate camera placement errors
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05), scale=(0.95, 1.05)),

    # 3. Photometric Augmentations (Simulating light source variations)
    # Saturation and Hue are kept very low to preserve the natural red blood color
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.1, hue=0.05),

    # 4. Convert to Tensor (Required before RandomErasing and Normalize)
    transforms.ToTensor(),

    # 5. Advanced Regularization: Random Erasing (Cutout)
    # Randomly masks a small patch (2% to 10% of the image) with black pixels (value=0).
    # This forces the network to learn the entire vascular network, preventing
    # it from relying on a single prominent vein or a shell defect.
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.1), ratio=(0.3, 3.3), value=0),

    # 6. Standard ImageNet Normalization
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Validation/Testing must NEVER be augmented (except for Resize and Normalize)
val_transforms = transforms.Compose([
    transforms.Resize((256, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = SegmentedEggDataset(data_list, transform=train_transforms)
val_dataset = SegmentedEggDataset(data_list, transform=val_transforms)
print(f"[*] Created Dataset with {len(train_dataset)} samples. Ready for K-Fold splitting.")
print(f"[*] Created Dataset with {len(val_dataset)} samples. Ready for K-Fold splitting.")


[*] Created Dataset with 4616 samples. Ready for K-Fold splitting.
[*] Created Dataset with 4616 samples. Ready for K-Fold splitting.


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
from torch.utils.data import Subset, DataLoader
from torchvision import models
import random
from collections import defaultdict
def get_fresh_vgg():
    """
    Creates a brand new VGG16 model with Batch Normalization.
    Pre-trained on ImageNet, modified for 2-class output.
    """
    # Load VGG16 with Batch Normalization
    model = models.vgg16_bn(weights=models.VGG16_BN_Weights.DEFAULT)

    # In VGG architectures, the classifier is a Sequential block.
    # The final output Linear layer is always located at index 6.
    num_features = model.classifier[6].in_features

    # Replace the final layer to output exactly 2 classes (Male/Female)
    model.classifier[6] = nn.Linear(num_features, 2)

    return model.to(device)

def get_custom_stratified_splits(labels, n_splits=5, seed=27):
    random.seed(seed)

    # Group indices by their class label (0 for Male, 1 for Female)
    class_indices = defaultdict(list)
    for idx, label in enumerate(labels):
        class_indices[label].append(idx)

    # Shuffle indices within each class to ensure randomness
    for label in class_indices:
        random.shuffle(class_indices[label])

    folds = [([], []) for _ in range(n_splits)] # List storing (train_idx, val_idx)

    # Distribute indices evenly across folds
    for label, indices in class_indices.items():
        chunk_size = len(indices) / n_splits
        for i in range(n_splits):
            start = int(i * chunk_size)
            end = int((i + 1) * chunk_size)

            val_chunk = indices[start:end]
            train_chunk = indices[:start] + indices[end:]

            folds[i][0].extend(train_chunk) # Add to train set
            folds[i][1].extend(val_chunk)   # Add to val set

    # Final shuffle to mix classes together in the DataLoader
    for i in range(n_splits):
        random.shuffle(folds[i][0])
        random.shuffle(folds[i][1])

    return folds

In [ ]:
import time
import copy
import torch
from tqdm.auto import tqdm

# ==========================================
# CORE TRAINING ENGINE
# ==========================================
def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler=None, num_epochs=20, patience=5, phase_name="Training"):

    # Track exactly how long the training takes
    since = time.time()

    # Store the best weights to revert to them later
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0

    # Variables for Early Stopping
    epochs_no_improve = 0
    early_stop = False

    # Check if GPU is available inside the function to ensure safety
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

    for epoch in range(num_epochs):
        # Break out of the main loop if early stopping was triggered
        if early_stop:
            print(f"\n🛑 EARLY STOPPING TRIGGERED! No improvement for {patience} epochs.")
            break

        print(f'\nEpoch {epoch+1}/{num_epochs} [{phase_name}]')
        print('-' * 15)

        # Each epoch has a training and validation phase
        for phase, loader in [('train', train_loader), ('val', val_loader)]:
            if phase == 'train':
                model.train()  # Set model to training mode
            else:
                model.eval()   # Set model to evaluate mode

            running_loss = 0.0
            running_corrects = 0
            processed_samples = 0

            # Get the total number of batches for this loader
            total_batches = len(loader)

            # Use enumerate to explicitly track the batch index.
            # unit="batch" tells tqdm to format speed as 'batches/s'
            pbar = tqdm(enumerate(loader), total=total_batches, leave=False, unit="batch")

            # Iterate over the data batches in the current loader
            for batch_idx, (inputs, labels) in pbar:

                # Dynamically update the description to show "Batch X/Y"
                pbar.set_description(f"{phase.capitalize()} Phase (Batch {batch_idx + 1}/{total_batches})")

                # Send data to GPU (if available) or CPU
                inputs = inputs.to(device)
                labels = labels.to(device)

                # Reset gradients before the forward pass
                optimizer.zero_grad()

                # Forward pass
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    # Backward pass & update weights ONLY in the 'train' phase
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                # Accumulate statistics
                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)
                processed_samples += inputs.size(0)

                # Update the progress bar postfix dynamically with the running loss
                current_live_loss = running_loss / processed_samples
                pbar.set_postfix({'Loss': f"{current_live_loss:.4f}"})

            # Step the learning rate scheduler at the end of the train phase
            # if phase == 'train' and scheduler is not None:
            #     scheduler.step()

            # Calculate average loss and accuracy for this epoch
            epoch_loss = running_loss / len(loader.dataset)
            epoch_acc = running_corrects.double() / len(loader.dataset)

            # Print results. If in training, also print the current Learning Rate.
            if phase == 'train':
                current_lr = optimizer.param_groups[0]['lr']
                print(f'{phase.capitalize()} | Loss: {epoch_loss:.4f} | Acc: {epoch_acc:.4f} | LR: {current_lr:.6f}')
            else:
                print(f'{phase.capitalize()}   | Loss: {epoch_loss:.4f} | Acc: {epoch_acc:.4f}')

            # ----------------------------------------------
            # VALIDATION LOGIC & EARLY STOPPING CHECK
            # ----------------------------------------------
            if phase == 'val':
                if epoch_acc > best_acc:
                    best_acc = epoch_acc
                    best_model_wts = copy.deepcopy(model.state_dict())
                    epochs_no_improve = 0
                    print(f"   🌟 New best accuracy achieved: {best_acc:.4f}")
                else:
                    epochs_no_improve += 1

                if epochs_no_improve >= patience:
                    early_stop = True
            if phase == 'val' and scheduler is not None:
                scheduler.step(epoch_loss)
    # Training is entirely finished
    time_elapsed = time.time() - since
    print(f'\n{phase_name} complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    print(f'Highest Validation Accuracy: {best_acc:.4f}')

    # Load the best model weights back into the model before returning it
    model.load_state_dict(best_model_wts)

    return model, best_acc

In [ ]:
# VGG model
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
from torch.utils.data import DataLoader, Subset
from tqdm.auto import tqdm

# ==========================================
# K-FOLD CONFIGURATION
# ==========================================
K_FOLDS = 5
CHECKPOINT_DIR = r"/content/drive/MyDrive/Major_Project/checkpoints/vgg"

# ⚠️ VGG16 uses much more VRAM than EfficientNet.
# If you get a "CUDA Out of Memory" error, reduce batch_size to 8, 4, or even 2.
batch_size = 32

print("="*50)
print(f"STARTING {K_FOLDS}-FOLD CROSS VALIDATION (VGG16-BN)")
print("="*50)

fold_results = {}

# Generate our custom, crash-free splits
# (Assuming get_custom_stratified_splits and all_labels are already defined)
custom_splits = get_custom_stratified_splits(all_labels, n_splits=K_FOLDS)


# Wrap the K-Fold loop with tqdm to create the progress bar
fold_pbar = tqdm(enumerate(custom_splits), total=K_FOLDS, desc="K-Fold Progress", unit="fold")

from collections import Counter

######################################################################################################
# Tự động đếm số lượng Trống/Mái trong toàn bộ dataset
label_counts = Counter(all_labels)
total_samples = len(all_labels)

# Tính trọng số nghịch đảo: Class nào càng ít ảnh, trọng số càng cao
weight_0 = total_samples / (2.0 * label_counts[0])
weight_1 = total_samples / (2.0 * label_counts[1])

# Đưa trọng số vào GPU
# This 'device' variable should be defined globally or passed into the function if not already available.
# Assuming 'device' is already defined in the global scope from previous cells or will be defined.
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
class_weights = torch.tensor([weight_0, weight_1], dtype=torch.float).to(device)

tqdm.write(f"[*] Auto-calculated Class Weights: Class 0: {weight_0:.2f}, Class 1: {weight_1:.2f}")
####################################################################################################

for fold, (train_idx, val_idx) in fold_pbar:
    # Use tqdm.write() instead of print() to prevent the progress bar from breaking visually
    tqdm.write(f"\n🚀 RUNNING FOLD {fold + 1}/{K_FOLDS}")
    tqdm.write("-" * 30)

    # 1. Create Subsets (CRITICAL: Use the FULL datasets here to prevent Index errors)
    train_sub = Subset(train_dataset, train_idx)
    val_sub = Subset(val_dataset, val_idx)

    # 2. Create DataLoaders
    train_loader = DataLoader(train_sub, batch_size=batch_size, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_sub, batch_size=batch_size, shuffle=False, num_workers=0)

    # 3. Initialize Fresh VGG Model
    model = get_fresh_vgg()
    criterion = nn.CrossEntropyLoss(weight=class_weights)

    optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

    # 4. Train Model
    trained_model, best_acc = train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        optimizer=optimizer,
        scheduler=scheduler,
        num_epochs=30,
        patience=5,
        phase_name=f"VGG Train Fold {fold+1}"
    )

    # 5. Save Model and Record Results
    fold_filename = f"vgg16_bn_fold_{fold+1}.pth"
    # Ensure CHECKPOINT_DIR exists before saving
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    fold_model_path = os.path.join(CHECKPOINT_DIR, fold_filename)

    torch.save(trained_model.state_dict(), fold_model_path)
    tqdm.write(f"[+] Saved best model for Fold {fold+1} to: {fold_model_path}")

    # Safely extract the float value to prevent numpy crashes
    safe_best_acc = best_acc.item() if torch.is_tensor(best_acc) else best_acc
    fold_results[fold] = safe_best_acc

    # Update the progress bar to show the best accuracy of the current fold
    fold_pbar.set_postfix({'Latest Acc': f"{safe_best_acc:.4f}"})

# ==========================================
# FINAL REPORT
# ==========================================
print("\n" + "="*50)
print("🏆 VGG16 K-FOLD CROSS VALIDATION RESULTS 🏆")
print("="*50)
for fold, acc in fold_results.items():
    print(f" - Fold {fold+1} Accuracy: {acc:.4f}")

average_acc = np.mean(list(fold_results.values()))
print(f"\n=> AVERAGE VALIDATION ACCURACY: {average_acc:.4f}")
print("="*50)


STARTING 5-FOLD CROSS VALIDATION (VGG16-BN)


K-Fold Progress:   0%|          | 0/5 [00:00<?, ?fold/s]

[*] Auto-calculated Class Weights: Class 0: 1.30, Class 1: 0.81

🚀 RUNNING FOLD 1/5
------------------------------

Epoch 1/30 [VGG Train Fold 1]
---------------


  0%|          | 0/116 [00:00<?, ?batch/s]

Train | Loss: 0.7046 | Acc: 0.5038 | LR: 0.000100


...

Val   | Loss: 0.6957 | Acc: 0.3853

Epoch 8/30 [VGG Train Fold 5]
---------------


  0%|          | 0/116 [00:00<?, ?batch/s]

Train | Loss: 0.6941 | Acc: 0.5081 | LR: 0.000050


  0%|          | 0/29 [00:00<?, ?batch/s]

Val   | Loss: 0.6940 | Acc: 0.4838

🛑 EARLY STOPPING TRIGGERED! No improvement for 5 epochs.

VGG Train Fold 5 complete in 19m 24s
Highest Validation Accuracy: 0.6147
[+] Saved best model for Fold 5 to: /content/drive/MyDrive/Major_Project/checkpoints/vgg/vgg16_bn_fold_5.pth

🏆 VGG16 K-FOLD CROSS VALIDATION RESULTS 🏆
 - Fold 1 Accuracy: 0.6150
 - Fold 2 Accuracy: 0.6143
 - Fold 3 Accuracy: 0.6093
 - Fold 4 Accuracy: 0.6143
 - Fold 5 Accuracy: 0.6147

=> AVERAGE VALIDATION ACCURACY: 0.6135


In [ ]:
import os
from torch.utils.data import DataLoader, random_split

# 1. Define the path to the filtered blood vessel dataset
bv_dir = r"/content/drive/MyDrive/Major_Project/filtered_dataset_blood_vessel_ori"

# 2. Extract image paths and labels into a list (similar to 'clean_data')
bv_data = []
class_mapping = {'0_Male': 0, '1_Female': 1}

print("SCANNING BLOOD VESSEL DIRECTORY...\n" + "-"*40)
for class_folder, label_idx in class_mapping.items():
    folder_path = os.path.join(bv_dir, class_folder)

    if not os.path.exists(folder_path):
        print(f"Warning: {folder_path} does not exist.")
        continue

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
            img_path = os.path.join(folder_path, filename)
            bv_data.append((img_path, label_idx))

print(f"Total blood vessel images found: {len(bv_data)}")

SCANNING BLOOD VESSEL DIRECTORY...
----------------------------------------
Total blood vessel images found: 186


In [ ]:
import os
import gc  # BẮT BUỘC TRÊN COLAB: Thư viện dọn rác bộ nhớ
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Subset, DataLoader
from torchvision import models
import copy
from tqdm.auto import tqdm
from collections import Counter

# ==========================================
# 0. CONFIGURATION & PATHS (DÀNH CHO COLAB)
# ==========================================
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"[*] Đang sử dụng thiết bị: {device}")

PHASE1_WEIGHTS_PATH = r"/content/drive/MyDrive/Major_Project/checkpoints/vgg/vgg16_bn_fold_1.pth"
PHASE2_SAVE_DIR = r"/content/drive/MyDrive/Major_Project/checkpoints_bv/"
os.makedirs(PHASE2_SAVE_DIR, exist_ok=True)

print("="*50)
print("STARTING PHASE 2: 5-FOLD FINE-TUNING (VGG16-BN - BLOOD VESSELS)")
print("="*50)

# ==========================================
# 1. PREPARE BLOOD VESSEL DATASET FOR K-FOLD
# ==========================================
full_bv_train_dataset = SegmentedEggDataset(bv_data, transform=train_transforms)
full_bv_val_dataset = SegmentedEggDataset(bv_data, transform=val_transforms)

bv_labels = [item[1] for item in bv_data]

K_FOLDS = 5
bv_custom_splits = get_custom_stratified_splits(bv_labels, n_splits=K_FOLDS)

# Tự động tính Class Weights cho VGG (Chống đoán lụi)
label_counts = Counter(bv_labels)
total_samples = len(bv_labels)
weight_0 = total_samples / (2.0 * label_counts.get(0, 1))
weight_1 = total_samples / (2.0 * label_counts.get(1, 1))
class_weights = torch.tensor([weight_0, weight_1], dtype=torch.float).to(device)
print(f"[*] Class Weights tự động: Class 0 (Trống): {weight_0:.2f}, Class 1 (Mái): {weight_1:.2f}")

# ==========================================
# 2. HELPER: LOAD PHASE 1 VGG BASELINE MODEL
# ==========================================
def get_phase1_vgg_baseline():
    """
    Khởi tạo VGG16_BN, nạp trọng số Phase 1, và reset lớp cuối.
    """
    model = models.vgg16_bn(weights=models.VGG16_BN_Weights.DEFAULT)

    # Lấy số features của lớp cuối cùng của VGG (vị trí số 6)
    num_features = model.classifier[6].in_features

    # BƯỚC 1: Xây dựng lại cấu trúc hệt như lúc lưu ở Phase 1 (Chỉ đổi lớp số 6)
    model.classifier[6] = nn.Linear(num_features, 2)

    # BƯỚC 2: Tải trọng số từ file .pth của VGG Phase 1
    model.load_state_dict(torch.load(PHASE1_WEIGHTS_PATH, map_location=device))

    # BƯỚC 3: RESET lớp classifier cuối cùng thành "trang giấy trắng"
    # Giữ nguyên phần Feature Extractor (thân mạng), chỉ reset cái "Đầu" đoán
    model.classifier[6] = nn.Linear(num_features, 2)

    return model.to(device)

# ==========================================
# 3. THE PHASE 2 K-FOLD LOOP
# ==========================================
phase2_fold_results = {}

# TRÊN COLAB: Nếu bị tràn RAM, hạ số này xuống 8.
batch_size = 16

fold_pbar = tqdm(enumerate(bv_custom_splits), total=K_FOLDS, desc="Phase 2 Fine-Tuning", unit="fold")

for fold, (train_idx, val_idx) in fold_pbar:
    tqdm.write(f"\n🔬 PHASE 2 - RUNNING FOLD {fold + 1}/{K_FOLDS}")
    tqdm.write("-" * 30)

    bv_train_sub = Subset(full_bv_train_dataset, train_idx)
    bv_val_sub = Subset(full_bv_val_dataset, val_idx)

    bv_train_loader = DataLoader(bv_train_sub, batch_size=batch_size, shuffle=True, num_workers=0)
    bv_val_loader = DataLoader(bv_val_sub, batch_size=batch_size, shuffle=False, num_workers=0)

    # Khởi tạo mô hình VGG (không phải EfficientNet nữa)
    model_phase2 = get_phase1_vgg_baseline()

    # Sử dụng Class Weights đã tính tự động
    criterion = nn.CrossEntropyLoss(weight=class_weights)

    # OPTIMIZER: Vì đã reset head, ta dùng lr=1e-4 để nó từ từ khớp với thân mạng
    optimizer_phase2 = optim.AdamW(model_phase2.parameters(), lr=1e-4, weight_decay=1e-4)
    scheduler_phase2 = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer_phase2,
        mode='min',
        factor=0.5,
        patience=2
    )

    # Train Model
    trained_model, best_acc = train_model(
        model=model_phase2,
        train_loader=bv_train_loader,
        val_loader=bv_val_loader,
        criterion=criterion,
        optimizer=optimizer_phase2,
        scheduler=scheduler_phase2,
        num_epochs=35,
        patience=7,
        phase_name=f"VGG Fine-Tune Fold {fold+1}"
    )

    # Đổi tên file lưu trữ để nhận diện đây là VGG Phase 2
    fold_filename = f"vgg16_bn_phase2_bv_fold_{fold+1}.pth"
    fold_model_path = os.path.join(PHASE2_SAVE_DIR, fold_filename)
    torch.save(trained_model.state_dict(), fold_model_path)

    tqdm.write(f"[+] Đã lưu mô hình tốt nhất cho Fold {fold+1} tại '{fold_model_path}'")

    phase2_fold_results[fold] = best_acc.item() if torch.is_tensor(best_acc) else best_acc
    fold_pbar.set_postfix({'Best Acc': f"{phase2_fold_results[fold]:.4f}"})

    # ==========================================
    # QUAN TRỌNG NHẤT DÀNH CHO GOOGLE COLAB: XẢ VRAM
    # ==========================================
    del model_phase2
    del trained_model
    del optimizer_phase2
    del criterion
    del scheduler_phase2
    del bv_train_loader
    del bv_val_loader

    gc.collect()
    torch.cuda.empty_cache()
    # ==========================================

# ==========================================
# 4. FINAL PHASE 2 REPORT
# ==========================================
print("\n" + "="*50)
print("🏆 PHASE 2 (BLOOD VESSEL) K-FOLD RESULTS 🏆")
print("="*50)
for fold, acc in phase2_fold_results.items():
    print(f" - Fold {fold+1} Accuracy: {acc:.4f}")

average_acc = np.mean(list(phase2_fold_results.values()))
print(f"\n=> AVERAGE FINE-TUNED VALIDATION ACCURACY: {average_acc:.4f}")
print("="*50)

[*] Đang sử dụng thiết bị: cuda:0
STARTING PHASE 2: 5-FOLD FINE-TUNING (VGG16-BN - BLOOD VESSELS)
[*] Class Weights tự động: Class 0 (Trống): 1.55, Class 1 (Mái): 0.74


Phase 2 Fine-Tuning:   0%|          | 0/5 [00:00<?, ?fold/s]


🔬 PHASE 2 - RUNNING FOLD 1/5
------------------------------

Epoch 1/35 [VGG Fine-Tune Fold 1]
---------------


  0%|          | 0/10 [00:00<?, ?batch/s]

Train | Loss: 0.6938 | Acc: 0.5638 | LR: 0.000100


...

Val   | Loss: 0.7546 | Acc: 0.2895

Epoch 8/35 [VGG Fine-Tune Fold 5]
---------------


  0%|          | 0/10 [00:00<?, ?batch/s]

Train | Loss: 0.6847 | Acc: 0.4797 | LR: 0.000025


  0%|          | 0/3 [00:00<?, ?batch/s]

Val   | Loss: 0.7109 | Acc: 0.3947

🛑 EARLY STOPPING TRIGGERED! No improvement for 7 epochs.

VGG Fine-Tune Fold 5 complete in 1m 38s
Highest Validation Accuracy: 0.6842
[+] Đã lưu mô hình tốt nhất cho Fold 5 tại '/content/drive/MyDrive/Major_Project/checkpoints_bv/vgg16_bn_phase2_bv_fold_5.pth'

🏆 PHASE 2 (BLOOD VESSEL) K-FOLD RESULTS 🏆
 - Fold 1 Accuracy: 0.6757
 - Fold 2 Accuracy: 0.7838
 - Fold 3 Accuracy: 0.6757
 - Fold 4 Accuracy: 0.6757
 - Fold 5 Accuracy: 0.6842

=> AVERAGE FINE-TUNED VALIDATION ACCURACY: 0.6990


In [ ]:
import os
import gc
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
from torch.utils.data import DataLoader, Subset
from torchvision import models
from tqdm.auto import tqdm
from collections import Counter

# ==========================================
# 0. HELPER FUNCTION CHO SWIN TRANSFORMER
# ==========================================
def get_fresh_swin_t():
    """Tạo mô hình Swin Transformer (Tiny)"""
    model = models.swin_t(weights=models.Swin_T_Weights.DEFAULT)

    # Ở các dòng Transformer, lớp dự đoán cuối cùng gọi là '.head'
    num_ftrs = model.head.in_features

    # Thay thế thành 2 class (Trống/Mái) + Dropout chống Overfitting
    model.head = nn.Sequential(
        nn.Dropout(p=0.4, inplace=True),
        nn.Linear(num_ftrs, 2)
    )

    return model.to(device)

# ==========================================
# 1. K-FOLD CONFIGURATION
# ==========================================
K_FOLDS = 5
CHECKPOINT_DIR = r"/content/drive/MyDrive/Major_Project/checkpoints"

# Swin-T khá nhẹ, nhưng cơ chế Attention tốn VRAM. Hãy thử 16, nếu lỗi OOM thì hạ xuống 8.
batch_size = 16

print("="*50)
print(f"STARTING {K_FOLDS}-FOLD CROSS VALIDATION (SWIN TRANSFORMER - FROZEN)")
print("="*50)

fold_results = {}
custom_splits = get_custom_stratified_splits(all_labels, n_splits=K_FOLDS)

# Tự động tính Class Weights cho hàm Loss để chống lệch data
label_counts = Counter(all_labels)
total_samples = len(all_labels)
weight_0 = total_samples / (2.0 * label_counts[0])
weight_1 = total_samples / (2.0 * label_counts[1])
class_weights = torch.tensor([weight_0, weight_1], dtype=torch.float).to(device)
print(f"[*] Đã tính Class Weights: Class 0 (Trống): {weight_0:.2f}, Class 1 (Mái): {weight_1:.2f}")

# Wrap the K-Fold loop with tqdm
fold_pbar = tqdm(enumerate(custom_splits), total=K_FOLDS, desc="K-Fold Progress", unit="fold")

for fold, (train_idx, val_idx) in fold_pbar:
    tqdm.write(f"\n🚀 RUNNING FOLD {fold + 1}/{K_FOLDS}")
    tqdm.write("-" * 30)

    # Create Subsets (Dùng full dataset)
    train_sub = Subset(train_dataset, train_idx)
    val_sub = Subset(val_dataset, val_idx)

    # Create DataLoaders
    train_loader = DataLoader(train_sub, batch_size=batch_size, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_sub, batch_size=batch_size, shuffle=False, num_workers=0)

    # ==========================================
    # 2. KHỞI TẠO VÀ ĐÓNG BĂNG SWIN TRANSFORMER
    # ==========================================
    model = get_fresh_swin_t()

    # BƯỚC A: Đóng băng TOÀN BỘ mạng lưới (Freeze all)
    for param in model.parameters():
        param.requires_grad = False

    # BƯỚC B: Mở khóa lớp '.head' (Lớp dự đoán Trống/Mái)
    for param in model.head.parameters():
        param.requires_grad = True

    # BƯỚC C (Cực kỳ quan trọng): Mở khóa Stage cuối cùng của Swin (features[7])
    # Điều này cho phép Attention điều chỉnh lại góc nhìn để tập trung vào vân máu
    for param in model.features[7].parameters():
        param.requires_grad = True

    # ==========================================
    # 3. CẤU HÌNH LOSS, OPTIMIZER VÀ SCHEDULER
    # ==========================================
    criterion = nn.CrossEntropyLoss(weight=class_weights)

    # Chỉ truyền các tham số đã mở khóa (requires_grad=True) vào Optimizer
    # AdamW là CHÂN ÁI (bắt buộc) khi dùng Vision Transformers.
    trainable_params = filter(lambda p: p.requires_grad, model.parameters())
    optimizer = optim.AdamW(trainable_params, lr=1e-4, weight_decay=1e-4)

    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

    # ==========================================
    # 4. THỰC THI HUẤN LUYỆN
    # ==========================================
    trained_model, best_acc = train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        optimizer=optimizer,
        scheduler=scheduler,
        num_epochs=35, # Swin có thể cần nhiều thời gian để hội tụ hơn CNN một chút
        patience=10,
        phase_name=f"Swin-T Train Fold {fold+1}"
    )

    # Save Model
    fold_filename = f"swin_t_fold_{fold+1}.pth"
    fold_model_path = os.path.join(CHECKPOINT_DIR, fold_filename)

    torch.save(trained_model.state_dict(), fold_model_path)
    tqdm.write(f"[+] Saved best model cho Fold {fold+1}: {fold_filename}")

    safe_best_acc = best_acc.item() if torch.is_tensor(best_acc) else best_acc
    fold_results[fold] = safe_best_acc

    fold_pbar.set_postfix({'Latest Acc': f"{safe_best_acc:.4f}"})

    # ==========================================
    # XẢ VRAM (Chống CUDA Out of Memory)
    # ==========================================
    del model
    del trained_model
    del optimizer
    del criterion
    del scheduler
    del train_loader
    del val_loader

    gc.collect()
    torch.cuda.empty_cache()
    # ==========================================

# ==========================================
# FINAL REPORT
# ==========================================
print("\n" + "="*50)
print("🏆 SWIN-T (FROZEN) K-FOLD CROSS VALIDATION RESULTS 🏆")
print("="*50)
for fold, acc in fold_results.items():
    print(f" - Fold {fold+1} Accuracy: {acc:.4f}")

average_acc = np.mean(list(fold_results.values()))
print(f"\n=> AVERAGE VALIDATION ACCURACY: {average_acc:.4f}")
print("="*50)

STARTING 5-FOLD CROSS VALIDATION (SWIN TRANSFORMER - FROZEN)


NameError: name 'get_custom_stratified_splits' is not defined

In [ ]:
import os
import gc  # BẮT BUỘC TRÊN COLAB: Thư viện dọn rác bộ nhớ
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Subset, DataLoader
from torchvision import models
import copy
from tqdm.auto import tqdm
from collections import Counter

# ==========================================
# 0. CONFIGURATION & PATHS
# ==========================================
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"[*] Đang sử dụng thiết bị: {device}")

# Đường dẫn trỏ đến file trọng số Swin-T Phase 1
PHASE1_WEIGHTS_PATH = r"/content/drive/MyDrive/Major_Project/checkpoints/swin/swin_t_fold_1.pth"
PHASE2_SAVE_DIR = r"/content/drive/MyDrive/Major_Project/checkpoints_bv"
os.makedirs(PHASE2_SAVE_DIR, exist_ok=True)

print("="*50)
print("STARTING PHASE 2: 5-FOLD FINE-TUNING (SWIN-T - BLOOD VESSELS)")
print("="*50)

# ==========================================
# 1. PREPARE BLOOD VESSEL DATASET FOR K-FOLD
# ==========================================
full_bv_train_dataset = SegmentedEggDataset(bv_data, transform=train_transforms)
full_bv_val_dataset = SegmentedEggDataset(bv_data, transform=val_transforms)

bv_labels = [item[1] for item in bv_data]

K_FOLDS = 5
bv_custom_splits = get_custom_stratified_splits(bv_labels, n_splits=K_FOLDS)

# Tự động tính Class Weights (Chống đoán lụi)
label_counts = Counter(bv_labels)
total_samples = len(bv_labels)
weight_0 = total_samples / (2.0 * label_counts.get(0, 1))
weight_1 = total_samples / (2.0 * label_counts.get(1, 1))
class_weights = torch.tensor([weight_0, weight_1], dtype=torch.float).to(device)
print(f"[*] Class Weights tự động: Class 0 (Trống): {weight_0:.2f}, Class 1 (Mái): {weight_1:.2f}")

# ==========================================
# 2. HELPER: LOAD PHASE 1 SWIN-T BASELINE MODEL
# ==========================================
def get_phase1_swin_baseline():
    """
    Khởi tạo Swin-T, nạp trọng số Phase 1, và reset lớp cuối.
    """
    model = models.swin_t(weights=models.Swin_T_Weights.DEFAULT)
    num_features = model.head.in_features

    # BƯỚC 1: Xây dựng lại cấu trúc hệt như lúc lưu ở Phase 1 (Sử dụng Sequential cho .head)
    model.head = nn.Sequential(
        nn.Dropout(p=0.4, inplace=True),
        nn.Linear(num_features, 2)
    )

    # BƯỚC 2: Tải trọng số từ file .pth của Swin-T Phase 1
    model.load_state_dict(torch.load(PHASE1_WEIGHTS_PATH, map_location=device))

    # BƯỚC 3: RESET lớp classifier cuối cùng thành "trang giấy trắng"
    # Giữ nguyên phần Thân mạng (Phase 1 knowledge), chỉ reset "Đầu" đoán để học bài toán mạch máu
    # model.head = nn.Sequential(
    #     nn.Dropout(p=0.4, inplace=True),
    #     nn.Linear(num_features, 2)
    # )

    return model.to(device)

# ==========================================
# 3. THE PHASE 2 K-FOLD LOOP
# ==========================================
phase2_fold_results = {}

# TRÊN COLAB: Nếu bị tràn RAM, hạ số này xuống 8.
batch_size = 16

fold_pbar = tqdm(enumerate(bv_custom_splits), total=K_FOLDS, desc="Phase 2 Fine-Tuning", unit="fold")

for fold, (train_idx, val_idx) in fold_pbar:
    tqdm.write(f"\n🔬 PHASE 2 - RUNNING FOLD {fold + 1}/{K_FOLDS}")
    tqdm.write("-" * 30)

    bv_train_sub = Subset(full_bv_train_dataset, train_idx)
    bv_val_sub = Subset(full_bv_val_dataset, val_idx)

    bv_train_loader = DataLoader(bv_train_sub, batch_size=batch_size, shuffle=True, num_workers=0)
    bv_val_loader = DataLoader(bv_val_sub, batch_size=batch_size, shuffle=False, num_workers=0)

    # 3.1 Khởi tạo mô hình Swin-T đã nạp tạ từ Phase 1
    model_phase2 = get_phase1_swin_baseline()

    # 3.2 ĐÓNG BĂNG MÔ HÌNH (PARTIAL FINE-TUNING)
    # Bước A: Khóa toàn bộ mạng
    for param in model_phase2.parameters():
        param.requires_grad = False

    # Bước B: Mở khóa lớp dự đoán cuối cùng (.head)
    for param in model_phase2.head.parameters():
        param.requires_grad = True

    # Bước C: Mở khóa Block Attention cuối (để thích nghi với hình ảnh mạch máu)
    for param in model_phase2.features[7].parameters():
        param.requires_grad = True

    # 3.3 Loss & Optimizer
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)

    # OPTIMIZER: Bắt buộc dùng filter để chỉ huấn luyện các lớp đã mở khóa (requires_grad=True)
    trainable_params = filter(lambda p: p.requires_grad, model_phase2.parameters())
    optimizer_phase2 = optim.AdamW(trainable_params, lr=1e-3, weight_decay=1e-4)

    scheduler_phase2 = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer_phase2,
        mode='min',
        factor=0.5,
        patience=2
    )

    # 3.4 Train Model
    trained_model, best_acc = train_model(
        model=model_phase2,
        train_loader=bv_train_loader,
        val_loader=bv_val_loader,
        criterion=criterion,
        optimizer=optimizer_phase2,
        scheduler=scheduler_phase2,
        num_epochs=40,
        patience=99,
        phase_name=f"Swin-T Phase 2 Fold {fold+1}"
    )

    # Đổi tên file lưu trữ để nhận diện đây là Swin-T Phase 2
    fold_filename = f"swin_t_phase2_bv_fold_{fold+1}.pth"
    fold_model_path = os.path.join(PHASE2_SAVE_DIR, fold_filename)
    torch.save(trained_model.state_dict(), fold_model_path)

    tqdm.write(f"[+] Đã lưu mô hình tốt nhất cho Fold {fold+1} tại '{fold_model_path}'")

    phase2_fold_results[fold] = best_acc.item() if torch.is_tensor(best_acc) else best_acc
    fold_pbar.set_postfix({'Best Acc': f"{phase2_fold_results[fold]:.4f}"})

    # ==========================================
    # QUAN TRỌNG NHẤT DÀNH CHO GOOGLE COLAB: XẢ VRAM
    # ==========================================
    del model_phase2
    del trained_model
    del optimizer_phase2
    del criterion
    del scheduler_phase2
    del bv_train_loader
    del bv_val_loader

    gc.collect()
    torch.cuda.empty_cache()
    # ==========================================

# ==========================================
# 4. FINAL PHASE 2 REPORT
# ==========================================
print("\n" + "="*50)
print(" PHASE 2 (BLOOD VESSEL) K-FOLD RESULTS WITH SWIN-T ")
print("="*50)
for fold, acc in phase2_fold_results.items():
    print(f" - Fold {fold+1} Accuracy: {acc:.4f}")

average_acc = np.mean(list(phase2_fold_results.values()))
print(f"\n=> AVERAGE FINE-TUNED VALIDATION ACCURACY: {average_acc:.4f}")
print("="*50)

[*] Đang sử dụng thiết bị: cuda:0
STARTING PHASE 2: 5-FOLD FINE-TUNING (SWIN-T - BLOOD VESSELS)
[*] Class Weights tự động: Class 0 (Trống): 1.55, Class 1 (Mái): 0.74


Phase 2 Fine-Tuning:   0%|          | 0/5 [00:00<?, ?fold/s]


🔬 PHASE 2 - RUNNING FOLD 1/5
------------------------------

Epoch 1/40 [Swin-T Phase 2 Fold 1]
---------------


  0%|          | 0/10 [00:00<?, ?batch/s]

Train | Loss: 0.8978 | Acc: 0.4698 | LR: 0.001000


...

Val   | Loss: 0.7074 | Acc: 0.5789

Epoch 40/40 [Swin-T Phase 2 Fold 5]
---------------


  0%|          | 0/10 [00:00<?, ?batch/s]

Train | Loss: 0.6846 | Acc: 0.5946 | LR: 0.000000


  0%|          | 0/3 [00:00<?, ?batch/s]

Val   | Loss: 0.7074 | Acc: 0.6053

Swin-T Phase 2 Fold 5 complete in 6m 20s
Highest Validation Accuracy: 0.6842
[+] Đã lưu mô hình tốt nhất cho Fold 5 tại '/content/drive/MyDrive/Major_Project/checkpoints_bv/swin_t_phase2_bv_fold_5.pth'

 PHASE 2 (BLOOD VESSEL) K-FOLD RESULTS WITH SWIN-T 
 - Fold 1 Accuracy: 0.7568
 - Fold 2 Accuracy: 0.6757
 - Fold 3 Accuracy: 0.6757
 - Fold 4 Accuracy: 0.6757
 - Fold 5 Accuracy: 0.6842

=> AVERAGE FINE-TUNED VALIDATION ACCURACY: 0.6936


In [ ]:
### MERGE ALL THE DATA AND BLOOD VESSEL
import os
import gc
import copy
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms, models
from PIL import Image
from tqdm.auto import tqdm
from collections import Counter
from sklearn.model_selection import StratifiedKFold

# ==========================================
# 0. CONFIGURATION & DEVICE SETUP
# ==========================================
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"[*] Using device: {device}")

SAVE_DIR = r"/content/drive/MyDrive/Major_Project/checkpoints_combined_vgg"
os.makedirs(SAVE_DIR, exist_ok=True)
K_FOLDS = 5
BATCH_SIZE = 16 # Adjust to 8 if you run into memory limits
NUM_EPOCHS = 25

# ==========================================
# 1. GATHERING DATA (Your provided logic)
# ==========================================
# -- CSV Data (RGB) --
CSV_PATH = r"/content/drive/MyDrive/Major_Project/linked_egg_metadata_qa_checked.csv"
IMAGE_DIR = r"/content/drive/MyDrive/Major_Project/merged_dataset_rgb"
df = pd.read_csv(CSV_PATH)
clean_df = df[df['QA_Failed'] == False].copy() if 'QA_Failed' in df.columns else df.copy()

LABEL_MAP = {'T': 0, 'M': 1}
data_list = []
for _, row in clean_df.iterrows():
    filename = os.path.basename(str(row['image_path']))
    img_path = os.path.join(IMAGE_DIR, filename)
    str_label = str(row['label']).strip().upper()
    if str_label in LABEL_MAP and os.path.exists(img_path):
        data_list.append((img_path, LABEL_MAP[str_label]))

# -- Folder Data (Blood Vessels) --
bv_dir = r"/content/drive/MyDrive/Major_Project/filtered_dataset_blood_vessel_ori"
bv_data = []
class_mapping = {'0_Male': 0, '1_Female': 1}
for class_folder, label_idx in class_mapping.items():
    folder_path = os.path.join(bv_dir, class_folder)
    if os.path.exists(folder_path):
        for filename in os.listdir(folder_path):
            if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
                bv_data.append((os.path.join(folder_path, filename), label_idx))

# ==========================================
# 2. MERGING THE DATASETS
# ==========================================
# Concatenate the two lists together
combined_data = data_list + bv_data

# Extract all labels for Stratified K-Fold splitting
combined_labels = [item[1] for item in combined_data]

print("="*50)
print(f"[*] Total RGB CSV Images: {len(data_list)}")
print(f"[*] Total Blood Vessel Images: {len(bv_data)}")
print(f"[*] TOTAL COMBINED IMAGES FOR TRAINING: {len(combined_data)}")
print("="*50)

[*] Using device: cuda:0
[*] Total RGB CSV Images: 4616
[*] Total Blood Vessel Images: 186
[*] TOTAL COMBINED IMAGES FOR TRAINING: 4802


In [ ]:
import os
import gc
import copy
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from tqdm.auto import tqdm
from collections import Counter
from sklearn.model_selection import train_test_split

# ==========================================
# 0. CONFIGURATION & DEVICE SETUP
# ==========================================
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"[*] Using device: {device}")

# Model option: 'vgg16' or 'convnext'
MODEL_TYPE = 'vgg16'

SAVE_DIR = r"/content/drive/MyDrive/Major_Project/checkpoints_single_run"
os.makedirs(SAVE_DIR, exist_ok=True)
BATCH_SIZE = 16
NUM_EPOCHS = 40

print("="*50)
print(f"STARTING TRAINING (TRAIN/VAL SPLIT) - MODEL: {MODEL_TYPE.upper()}")
print("="*50)

# ==========================================
# 1. GATHER DATA FROM 2 SOURCES
# ==========================================
# -- From CSV file (RGB) --
CSV_PATH = r"/content/drive/MyDrive/Major_Project/linked_egg_metadata_qa_checked.csv"
IMAGE_DIR = r"/content/drive/MyDrive/Major_Project/merged_dataset_rgb"
df = pd.read_csv(CSV_PATH)
clean_df = df[df['QA_Failed'] == False].copy() if 'QA_Failed' in df.columns else df.copy()

LABEL_MAP = {'T': 0, 'M': 1}
data_list = []
for _, row in clean_df.iterrows():
    filename = os.path.basename(str(row['image_path']))
    img_path = os.path.join(IMAGE_DIR, filename)
    str_label = str(row['label']).strip().upper()
    if str_label in LABEL_MAP and os.path.exists(img_path):
        data_list.append((img_path, LABEL_MAP[str_label]))

# -- From Folder (Blood Vessels) --
bv_dir = r"/content/drive/MyDrive/Major_Project/filtered_dataset_blood_vessel_ori"
bv_data = []
class_mapping = {'0_Male': 0, '1_Female': 1}
for class_folder, label_idx in class_mapping.items():
    folder_path = os.path.join(bv_dir, class_folder)
    if os.path.exists(folder_path):
        for filename in os.listdir(folder_path):
            if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
                bv_data.append((os.path.join(folder_path, filename), label_idx))

# -- Merge data --
combined_data = data_list + bv_data
combined_labels = [item[1] for item in combined_data]

print(f"[*] Total images: {len(combined_data)} (RGB: {len(data_list)}, Blood Vessels: {len(bv_data)})")

# ==========================================
# 2. TRAIN / VALIDATION SPLIT (80% / 20%)
# ==========================================
# Replacing K-Fold with standard data splitting.
# 'stratify=combined_labels' ensures an equal ratio of Male/Female in both sets.
train_data, val_data, train_labels, val_labels = train_test_split(
    combined_data,
    combined_labels,
    test_size=0.2,
    stratify=combined_labels,
    random_state=42
)

# Calculate Class Weights based on the Train set
label_counts = Counter(train_labels)
total_samples = len(train_labels)
weight_0 = total_samples / (2.0 * label_counts.get(0, 1))
weight_1 = total_samples / (2.0 * label_counts.get(1, 1))
class_weights = torch.tensor([weight_0, weight_1], dtype=torch.float).to(device)
print(f"[*] Auto-calculated Class Weights: Class 0: {weight_0:.2f}, Class 1: {weight_1:.2f}")

# ==========================================
# 3. DATASET CLASS & TRANSFORMS
# ==========================================
class CombinedEggDataset(Dataset):
    def __init__(self, data_list, transform=None):
        self.data_list = data_list
        self.transform = transform

    def __len__(self):
        return len(self.data_list)

    def __getitem__(self, idx):
        img_path, label = self.data_list[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform: image = self.transform(image)
        return image, label

train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = CombinedEggDataset(train_data, transform=train_transforms)
val_dataset = CombinedEggDataset(val_data, transform=val_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# ==========================================
# 4. MODEL INITIALIZATION AND ANTI-UNDERFITTING
# ==========================================
def get_model(model_type):
    if model_type == 'vgg16':
        model = models.vgg16_bn(weights=models.VGG16_BN_Weights.DEFAULT)
        for param in model.parameters(): param.requires_grad = False
        # Unfreeze Block 5 to learn better features
        for param in model.features[34:].parameters(): param.requires_grad = True

        num_features = model.classifier[6].in_features
        model.classifier[6] = nn.Sequential(
            nn.Dropout(p=0.3, inplace=True),
            nn.Linear(num_features, 2)
        )

    elif model_type == 'convnext':
        model = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT)
        for param in model.parameters(): param.requires_grad = False
        # Unfreeze the last stage
        for param in model.features[7].parameters(): param.requires_grad = True

        num_features = model.classifier[2].in_features
        model.classifier[2] = nn.Sequential(
            nn.Dropout(p=0.3, inplace=True),
            nn.Linear(num_features, 2)
        )
    return model.to(device)

model = get_model(MODEL_TYPE)

# ==========================================
# 5. SINGLE RUN TRAINING LOOP (WITH TQDM PROGRESS BARS)
# ==========================================
criterion = nn.CrossEntropyLoss(weight=class_weights)
trainable_params = filter(lambda p: p.requires_grad, model.parameters())

# Loosen configuration to make it easier for the model to learn
optimizer = optim.AdamW(trainable_params, lr=1e-4, weight_decay=1e-5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

best_acc = 0.0
best_model_wts = copy.deepcopy(model.state_dict())
epochs_no_improve = 0
patience_limit = 8

print(f"\n🚀 RUNNING TRAINING...")

for epoch in range(NUM_EPOCHS):
    # ---------- TRAINING ----------
    model.train()
    running_loss, running_corrects = 0.0, 0

    # Initialize the Training Progress Bar
    train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Train]", leave=False)

    for inputs, labels in train_pbar:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        running_corrects += torch.sum(preds == labels.data)

        # Update the progress bar with the current loss
        train_pbar.set_postfix({'loss': f"{loss.item():.4f}"})

    train_loss = running_loss / len(train_dataset)
    train_acc = running_corrects.double() / len(train_dataset)

    # ---------- VALIDATION ----------
    model.eval()
    val_loss, val_corrects = 0.0, 0

    # Initialize the Validation Progress Bar
    val_pbar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Val]", leave=False)

    with torch.no_grad():
        for inputs, labels in val_pbar:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * inputs.size(0)
            val_corrects += torch.sum(preds == labels.data)

            # Update the progress bar with the current loss
            val_pbar.set_postfix({'loss': f"{loss.item():.4f}"})

    val_loss = val_loss / len(val_dataset)
    val_acc = val_corrects.double() / len(val_dataset)

    scheduler.step(val_loss)

    print(f"Epoch {epoch+1:02d}/{NUM_EPOCHS} | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")

    # Evaluate and Early Stopping
    if val_acc > best_acc:
        best_acc = val_acc
        best_model_wts = copy.deepcopy(model.state_dict())
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1

    if epochs_no_improve >= patience_limit:
        print(f"[*] Early stopping triggered at Epoch {epoch + 1}!")
        break

# ---------- SAVE MODEL AND CLEANUP ----------
model_path = os.path.join(SAVE_DIR, f"best_{MODEL_TYPE}_model.pth")
torch.save(best_model_wts, model_path)
print("\n" + "="*50)
print(f"🎉 TRAINING COMPLETE. Best Accuracy (Validation): {best_acc:.4f}")
print(f"💾 Model saved at: {model_path}")
print("="*50)

del model, train_loader, val_loader, optimizer, criterion, scheduler
gc.collect()
torch.cuda.empty_cache()

[*] Using device: cuda:0
STARTING TRAINING (TRAIN/VAL SPLIT) - MODEL: VGG16
[*] Total images: 4802 (RGB: 4616, Blood Vessels: 186)
[*] Auto-calculated Class Weights: Class 0: 1.31, Class 1: 0.81

🚀 RUNNING TRAINING...


Epoch 1/40 [Train]:   0%|          | 0/241 [00:00<?, ?it/s]

Epoch 1/40 [Val]:   0%|          | 0/61 [00:00<?, ?it/s]

Epoch 01/40 | Train Loss: 0.7071 Acc: 0.5103 | Val Loss: 0.6933 Acc: 0.5973


Epoch 2/40 [Train]:   0%|          | 0/241 [00:00<?, ?it/s]

...

Epoch 8/40 [Val]:   0%|          | 0/61 [00:00<?, ?it/s]

Epoch 08/40 | Train Loss: 0.6739 Acc: 0.5931 | Val Loss: 0.7076 Acc: 0.5088


Epoch 9/40 [Train]:   0%|          | 0/241 [00:00<?, ?it/s]

Epoch 9/40 [Val]:   0%|          | 0/61 [00:00<?, ?it/s]

Epoch 09/40 | Train Loss: 0.6699 Acc: 0.5957 | Val Loss: 0.7193 Acc: 0.5297
[*] Early stopping triggered at Epoch 9!

🎉 TRAINING COMPLETE. Best Accuracy (Validation): 0.5973
💾 Model saved at: /content/drive/MyDrive/Major_Project/checkpoints_single_run/best_vgg16_model.pth
